# Survival Modeling
## Prediction of overall survival of patients with Myeloid Leukemia

Objective: modeling the survivability of the patients using the clinical dataset.

## Importing Modules

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# model selection
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV

# my module
from leukemia.models import evaluation
from leukemia.data import loader
from leukemia.models import evaluation

# preprocessing and pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import  make_pipeline

# LGBM
from lightgbm import LGBMRegressor

# Cox models
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import as_concordance_index_ipcw_scorer

# ttest
from scipy.stats import ttest_rel


## Importing data

In [ ]:
X, y = loader.split_clinical_processed("../data/processed/clinical_train_processed_v1.csv")

num_cols = ['BM_BLAST_LOG', 'WBC_LOG', 'ANC_LOG', 'MONOCYTES_LOG', 'HB',
       'PLT_LOG']
binary_cols = ['BM_BLAST_MISSING', 'WBC_MISSING',
       'ANC_MISSING','MONOCYTES_MISSING', 'HB_MISSING', 'PLT_MISSING',
       'CYTO_NORMAL', 'CYTO_COMPLEX', 'CYTO_ABNORMAL', 'CYTO_MISSING']

## Splitting data training and testing sets

We are splitting the dataset to keep 25% of the data for the final test of our model. We are setting shuffle to True as the dataset is sorted by center.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, shuffle=True)

## 1. Baseline: LightGBM

We are doing the cross validation manually as the C-index takes a structured array and needs the y_train set to evaluate the proportion of censoring. The LGB, however, takes a pandas dataframe or a single column array. This makes the use of cross_validate a little bit tricky.

In [ ]:
preprocess_LGB = ColumnTransformer([
    ("num", "passthrough", num_cols),("binary", "passthrough", binary_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER']),
])

mod_pipe_LGB = LGBMRegressor()

pipe_LGB = make_pipeline(preprocess_LGB, mod_pipe_LGB)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scorer = evaluation.ipcw_c_index
scores = []

for fold, (train_index, val_index) in enumerate(cv.split(X_train, y_train["OS_STATUS"])):
    X_train_cv, X_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_train_cv, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

    pipe_LGB.fit(X_train_cv, y_train_cv['OS_YEARS'])

    y_train_cv = loader.make_surv_target(y_train_cv['OS_YEARS'], y_train_cv['OS_STATUS'])
    y_val = loader.make_surv_target(y_val['OS_YEARS'], y_val['OS_STATUS'])

    scores.append({"fold": fold, "train_sore": scorer(y_train_cv, y_train_cv, -pipe_LGB.predict(X_train_cv)),
    "val_sore": scorer(y_train_cv, y_val, -pipe_LGB.predict(X_val))})

scores = pd.DataFrame(scores)
fold_scores_baseline_LGB = scores['val_sore']
print(scores, scores["val_sore"].mean(), scores["val_sore"].std())


The LGBM shows a c-index of 0.5997 ± 0.022. The LGBM does not performed better than our BM_BLAST baseline (0.616 ± 0.009). The LGBM does not take the censoring into account and therefore consider a deceased patient at 2 year as a censored patient at 2 years.

## 2. First COX regression model

The protocol require 2 adjustments :
- the StratifiedKFold must stratify on OS_STATUS
- the c-index must have OS_STATUS from y_train
We are developing a method to adapt the protocol.

In [ ]:
def run_cv_cox(pipeline, X_train: pd.DataFrame, y_train: pd.DataFrame, cv: StratifiedKFold, scorer) -> tuple :
    """
    Evaluate the cox model
    :param model: model we want to evaluate
    :param X: feature dataset
    :param y: target dataset
    :param cv: cross validation split
    :param scorer: scoring function: IPCW C-index
    :return: the cv results (n°fold, train_sore, val_sore) and the mean score of the validation sets
    """
    scores = []
    for fold, (train_index, val_index) in enumerate(cv.split(X_train, y_train["OS_STATUS"])):
        X_train_cv, X_val = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_cv, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

        y_train_cv = loader.make_surv_target(y_train_cv['OS_YEARS'], y_train_cv['OS_STATUS'])
        y_val = loader.make_surv_target(y_val['OS_YEARS'], y_val['OS_STATUS'])

        pipeline.fit(X_train_cv, y_train_cv)

        scores.append({"fold": fold, "train_sore": scorer(y_train_cv, y_train_cv, pipeline.predict(X_train_cv)),
        "val_sore": scorer(y_train_cv, y_val, pipeline.predict(X_val))})
        score_mean = {'mean': pd.DataFrame(scores)["val_sore"].mean(), 'std': pd.DataFrame(scores)["val_sore"].std()}

    return pd.DataFrame(scores), pd.DataFrame([score_mean])

First Cox regression: we are using the base prprocessed data set with:
- biological parameters transformed with log1p
- biological parameters missing values indicators
- cytogenetics split into 4 features:
    - normal
    - abnormal
    - missing
    - complex

We are crating a preprocess_base transformer:
- Standard scale the numerical features
- Replacing the missing values in the numerical features by the median
- One hot encoding the discrete feature CENTER


In [ ]:
preprocess_base = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols), ("binary", "passthrough", binary_cols), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER'])])

cv= evaluation.get_cv(5)
scorer = evaluation.ipcw_c_index
pipe_cox_base = make_pipeline(preprocess_base, CoxPHSurvivalAnalysis(alpha=1e-4))
fold_scores_cox_1, mean_score_cox_1 = run_cv_cox(pipe_cox_base, X_train, y_train, cv, scorer)
print(fold_scores_cox_1)
print(mean_score_cox_1)

Our reference score for the cox Regression is 0.6847 ± 0.0278. \
The score of the cox regression is higher than our BM_BLAST baseline (0.616 ± 0.009) and the LGBM baseline (0.599664 ± 0.022042). \
Lets realize a paired t-test to see if the difference is significant.

In [ ]:
res_cox_vs_baseline = ttest_rel(fold_scores_cox_1['val_sore'], fold_scores_baseline_LGB)
print(f"t = {res_cox_vs_baseline.statistic:.3f}, p = {res_cox_vs_baseline.pvalue:.4f}")

The t-test shows :
- t = 7.697
- p-value = 0.0015 \
The test confirms that our cox model is significantly better than our baseline.

## 3. Cox Regression on the features engineered CYTOGENETICS

### Importing the dataset

In [ ]:
X, y = loader.split_clinical_processed('../data/processed/clinical_train_feature_engin.csv')
num_cols = ['BM_BLAST_LOG', 'WBC_LOG', 'ANC_LOG', 'MONOCYTES_LOG', 'HB',
       'PLT_LOG']
binary_cols = ['BM_BLAST_MISSING', 'WBC_MISSING',
       'ANC_MISSING','MONOCYTES_MISSING', 'HB_MISSING', 'PLT_MISSING']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, shuffle=True)

We are crating a preprocess_v1 transformer:
- Standard scale the numerical features
- Replacing the missing values in the numerical features by the median
- One hot encoding the discrete features : CENTER and CYTO_LVL

In [ ]:
preprocess_v1 = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols), ("binary", "passthrough", binary_cols), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER', 'CYTO_LVL'])])

pipe_cox_v1 = make_pipeline(preprocess_v1, CoxPHSurvivalAnalysis(alpha=1e-4))

fold_scores_cox_2, mean_score_cox_2 = run_cv_cox(pipe_cox_v1, X_train, y_train, cv, scorer)
print(fold_scores_cox_2)
print(mean_score_cox_2)

res_coxv1_vs_coxv2 = ttest_rel(fold_scores_cox_2['val_sore'], fold_scores_cox_1['val_sore'])
print(f"t = {res_coxv1_vs_coxv2.statistic:.3f}, p = {res_coxv1_vs_coxv2.pvalue:.4f}")

Unfortunately the featured engineered version does not seem improve significantly the model. The c-index is 0.690717 ± 0.025865. \
The t-test shows a t of 2.498 and a p-value of 0.0669, which suggest a small improvement. We are keeping the engineered feature.

In [ ]:
coefs_pipe_cox_v1 = pd.Series(pipe_cox_v1[-1].coef_, index=pipe_cox_v1[0].get_feature_names_out()).sort_values()
coefs_pipe_cox_v1

The coefficients of the models show low values for the missing indicators, suggesting a low impact on the prediction. Some centers have prety high coefficients, both positive ande negatives. This is consistent with what we observer during the EDA, some centers have higher death or censoring rates tahn other. As expected BM_BLAST has a positive impact on the risk.
Moreover, as expected CYTO_LVL very poor and poor have a positive impact on the prediction. However, the CYTO_LVL_NORMAL feature show a more important negative impact on the global survivability than CYTO_LVL_GOOD and CYTO_LVL_VERY_GOOD. This may suggest that our feature fails to really classify the karyotypes.

## 4. Pruning the features

We are keeping the feature engineered dataset, but we are removing the missing features indicators, as they seem to bring low informations.

In [ ]:
X, y = loader.split_clinical_processed('../data/processed/clinical_train_feature_engin.csv')

X.drop(['BM_BLAST_MISSING', 'WBC_MISSING', 'ANC_MISSING', 'MONOCYTES_MISSING', 'HB_MISSING', 'PLT_MISSING'],axis=1, inplace=True)

num_cols = ['BM_BLAST_LOG', 'WBC_LOG', 'ANC_LOG', 'MONOCYTES_LOG', 'HB',
       'PLT_LOG']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
preprocess_v1 = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER', 'CYTO_LVL'])])

pipe_cox_v2 = make_pipeline(preprocess_v1, CoxPHSurvivalAnalysis(alpha=1e-4))

fold_scores_cox_3, mean_score_cox_3 = run_cv_cox(pipe_cox_v2, X_train, y_train, cv, scorer)
print(fold_scores_cox_3)
print(mean_score_cox_3)

In [ ]:
res_coxv2_vs_coxv3 = ttest_rel(fold_scores_cox_3['val_sore'],fold_scores_cox_2['val_sore'])
print(f"t = {res_coxv2_vs_coxv3.statistic:.3f}, p = {res_coxv2_vs_coxv3.pvalue:.4f}")

The model without the Missing indicators shows no significant performance improve or decrease: 0.6912 ± 0.0261. The t-test shows a t of 0.770 and a p-value of 0.4842. We are removing the missing indicators for the rest of our study.

## 5. Feature selection with Coxnet L1

In [ ]:
from sklearn.model_selection import GridSearchCV
from sksurv.metrics import as_concordance_index_ipcw_scorer
from sksurv.linear_model import CoxnetSurvivalAnalysis

In [ ]:
preprocess_cox_net = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER', 'CYTO_LVL'])])

pipe_coxnet = make_pipeline(preprocess_cox_net, CoxnetSurvivalAnalysis())
folds = cv.split(X_train, y_train['OS_STATUS'])
y_train_coxnet = loader.make_surv_target(y_train['OS_YEARS'], y_train['OS_STATUS'])

pipe_coxnet.fit(X_train, y_train_coxnet)
alphas = pipe_coxnet[-1].alphas_
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gcv = GridSearchCV(as_concordance_index_ipcw_scorer(pipe_coxnet, tau=7.0),
    param_grid={"estimator__coxnetsurvivalanalysis__alphas": [[a] for a in alphas]},
    cv=folds,
    error_score=0.5,
    n_jobs=4,
).fit(X_train, y_train_coxnet)

In [ ]:
idx = gcv.best_index_
cvr = gcv.cv_results_


best_alpha = cvr["params"][idx]["estimator__coxnetsurvivalanalysis__alphas"][0]
print("best alpha :", best_alpha)

per_fold = [cvr[f"split{i}_test_score"][idx] for i in range(5)]
print("scores fold par fold :", per_fold)

alphas = [p["estimator__coxnetsurvivalanalysis__alphas"][0] for p in cvr["params"]]
means  = cvr["mean_test_score"]
print("alphas :", alphas)
print("scores :", means)

In [ ]:
plt.plot(alphas, means)
plt.title("Performance en fonction de alpha")
plt.xlabel("alpha")
plt.ylabel("score")
plt.show()

In [ ]:
alpha = 0.0025288732011953413

preprocess_cox_net = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER', 'CYTO_LVL'])])

pipe_coxnet = make_pipeline(preprocess_cox_net, CoxnetSurvivalAnalysis(alphas=[alpha]))

fold_scores_coxnet, mean_score_coxnet = run_cv_cox(pipe_coxnet, X_train, y_train, cv, scorer)
print(fold_scores_coxnet, mean_score_coxnet)


In [ ]:
res_coxv3_vs_coxnet = ttest_rel(fold_scores_coxnet['val_sore'],fold_scores_cox_3['val_sore'])
print(f"t = {res_coxv3_vs_coxnet.statistic:.3f}, p = {res_coxv3_vs_coxnet.pvalue:.4f}")

The régularisation Elastic Net shows a best alpha of 0.0025288732011953413. The new model train with this parameter show a C-index of 0.6952 ± 0.0295. \
The results of the ttest compared to the precedent model are:
- t = 2.197
- p = 0.0929 \
The regularization model shows no performance improvement.

In [ ]:
coefs_pipe_coxnet = pd.Series(
    pipe_coxnet[-1].coef_[:, np.argmin(np.abs(pipe_coxnet[-1].alphas_ - best_alpha))],
    index=pipe_coxnet[0].get_feature_names_out()
).sort_values()
coefs_pipe_coxnet

The regularization put to 0 the coefficients of several centers (REL, VU, ...) but also CYTO_LVL_MISSING and most surprizingly WBC_LOG.

## 6. Training of the final version of the model

In [ ]:
y_train = loader.make_surv_target(y_train['OS_YEARS'], y_train['OS_STATUS'])
y_test = loader.make_surv_target(y_test['OS_YEARS'], y_test['OS_STATUS'])

pipe_coxnet.fit(X_train, y_train)
risk_coxnet = pipe_coxnet.predict(X_test)
print(evaluation.ipcw_c_index(y_train, y_test, risk_coxnet))

The C-index on the test sets is 0.6856.

# Conclution

We were not able to pass the 0.7 bar with our currents model and features. Our best model performed 0.6856 for the final test, which still outperforme our BM_BLAST baseline.
- The missing indicators seems to not bring any informations even tho the missing values are not randomly missing.
- The coefficients for the CYTOGENETICS indicator shows that our feature may not be able to successfully separate the good, very good and intermediate karyotypes.

### Next steps:
- exploring more elaborate way of replacing missing values
- improving the CYTOGENETICS indicator feature
- exploring the molecular dataset
- trying a XGboost cox model